# MB52 — Centro 4014 — Datalake

**Tabela:** `dev_procurement.corp_curated.tbl_ds_log_mb52`
**Domínio:** Estoque por deposito
**Filtro do cenário:** `cod_centro = '4014'`
**Colunas:** 23 · **Clustering declarado:** `cod_material`, `cod_centro`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## Objetivo

Extrair e caracterizar **toda** a base do centro 4014 para comparação com o extrato do SAP.

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados da tabela |
| 2 | Volumetria e representatividade do cenário |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| 6 | Preenchimento de todas as colunas |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Perfil numérico |
| **10** | **Totais para conciliação com o SAP** |
| 11 | Datas |
| 12 | Códigos e zeros à esquerda |
| **13** | **Chaves normalizadas para join** |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| 18 | Análises específicas |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados da tabela

In [ ]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_log_mb52;

In [ ]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_log_mb52;

In [ ]:
-- Ultimas gravacoes (falha se for view)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_log_mb52 LIMIT 20;

## 2. Volumetria e representatividade

Quanto o centro 4014 representa do total da tabela.

In [ ]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mb52)                                   AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')                               AS linhas_centro_4014,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mb52), 4)                  AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro`) FROM dev_procurement.corp_curated.tbl_ds_log_mb52)                     AS centros_na_tabela;

## 3. Confirmação do filtro

Confirma que o valor `4014` existe e que não há variação de formato
(espaços, zeros à esquerda) que faça o filtro perder linhas silenciosamente.

**Se retornar mais de uma linha, o filtro `= '4014'` está incompleto.**

In [ ]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro` AS STRING)                    AS valor_bruto,
       length(CAST(`cod_centro` AS STRING))            AS comprimento,
       COUNT(*)                                   AS linhas
FROM dev_procurement.corp_curated.tbl_ds_log_mb52
WHERE regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') = '4014'
   OR trim(CAST(`cod_centro` AS STRING)) = '4014'
GROUP BY CAST(`cod_centro` AS STRING), length(CAST(`cod_centro` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [ ]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito` FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito + tp_estoque_especial' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial` FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito + tp_estoque_especial + dateingest' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial`, `dateingest` FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `cod_material + cod_centro`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [ ]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_material`, `cod_centro`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
GROUP BY `cod_material`, `cod_centro`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [ ]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
),
dup AS (
  SELECT `cod_material`, `cod_centro` FROM cen GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`cod_material` <=> dup.`cod_material` AND c.`cod_centro` <=> dup.`cod_centro`
),
agg AS (
  SELECT `cod_material`, `cod_centro`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `tp_material`) AS `tp_material`,
         COUNT(DISTINCT `tp_grupo_mercadorias`) AS `tp_grupo_mercadorias`,
         COUNT(DISTINCT `nm_centro`) AS `nm_centro`,
         COUNT(DISTINCT `ind_eliminacao_deposito`) AS `ind_eliminacao_deposito`,
         COUNT(DISTINCT `tp_estoque_especial`) AS `tp_estoque_especial`,
         COUNT(DISTINCT `qt_utilizacao_livre`) AS `qt_utilizacao_livre`,
         COUNT(DISTINCT `sg_unidade_medida_basica`) AS `sg_unidade_medida_basica`,
         COUNT(DISTINCT `vl_utilizacao_livre`) AS `vl_utilizacao_livre`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `qt_transito`) AS `qt_transito`,
         COUNT(DISTINCT `vl_transito`) AS `vl_transito`,
         COUNT(DISTINCT `qt_controle_qualidade`) AS `qt_controle_qualidade`,
         COUNT(DISTINCT `vl_controle_qualidade`) AS `vl_controle_qualidade`,
         COUNT(DISTINCT `qt_estoque_bloqueado`) AS `qt_estoque_bloqueado`,
         COUNT(DISTINCT `vl_estoque_bloqueado`) AS `vl_estoque_bloqueado`,
         COUNT(DISTINCT `cod_estoque_especial`) AS `cod_estoque_especial`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`,
         COUNT(DISTINCT `yearingest`) AS `yearingest`,
         COUNT(DISTINCT `monthingest`) AS `monthingest`
  FROM d GROUP BY `cod_material`, `cod_centro`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(21,
    'desc_material', MAX(`desc_material`),
    'cod_deposito', MAX(`cod_deposito`),
    'tp_material', MAX(`tp_material`),
    'tp_grupo_mercadorias', MAX(`tp_grupo_mercadorias`),
    'nm_centro', MAX(`nm_centro`),
    'ind_eliminacao_deposito', MAX(`ind_eliminacao_deposito`),
    'tp_estoque_especial', MAX(`tp_estoque_especial`),
    'qt_utilizacao_livre', MAX(`qt_utilizacao_livre`),
    'sg_unidade_medida_basica', MAX(`sg_unidade_medida_basica`),
    'vl_utilizacao_livre', MAX(`vl_utilizacao_livre`),
    'cod_moeda', MAX(`cod_moeda`),
    'qt_transito', MAX(`qt_transito`),
    'vl_transito', MAX(`vl_transito`),
    'qt_controle_qualidade', MAX(`qt_controle_qualidade`),
    'vl_controle_qualidade', MAX(`vl_controle_qualidade`),
    'qt_estoque_bloqueado', MAX(`qt_estoque_bloqueado`),
    'vl_estoque_bloqueado', MAX(`vl_estoque_bloqueado`),
    'cod_estoque_especial', MAX(`cod_estoque_especial`),
    'dateingest', MAX(`dateingest`),
    'yearingest', MAX(`yearingest`),
    'monthingest', MAX(`monthingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4014 — ou o contrário.
Por isso a varredura é feita sobre o recorte, não sobre a base completa.

In [ ]:
-- 6. PREENCHIMENTO NO CENTRO 4014
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'),
perf AS (
  SELECT stack(23,
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'tp_material', 'string', COUNT_IF(`tp_material` IS NULL), COUNT_IF(`tp_material` IS NOT NULL AND lower(trim(`tp_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_material`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_mercadorias', 'string', COUNT_IF(`tp_grupo_mercadorias` IS NULL), COUNT_IF(`tp_grupo_mercadorias` IS NOT NULL AND lower(trim(`tp_grupo_mercadorias`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_mercadorias`) RLIKE '^0+([.,]0+)?$'),
    'nm_centro', 'string', COUNT_IF(`nm_centro` IS NULL), COUNT_IF(`nm_centro` IS NOT NULL AND lower(trim(`nm_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_centro`) RLIKE '^0+([.,]0+)?$'),
    'ind_eliminacao_deposito', 'string', COUNT_IF(`ind_eliminacao_deposito` IS NULL), COUNT_IF(`ind_eliminacao_deposito` IS NOT NULL AND lower(trim(`ind_eliminacao_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_eliminacao_deposito`) RLIKE '^0+([.,]0+)?$'),
    'tp_estoque_especial', 'string', COUNT_IF(`tp_estoque_especial` IS NULL), COUNT_IF(`tp_estoque_especial` IS NOT NULL AND lower(trim(`tp_estoque_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_estoque_especial`) RLIKE '^0+([.,]0+)?$'),
    'qt_utilizacao_livre', 'decimal(13,3)', COUNT_IF(`qt_utilizacao_livre` IS NULL), 0L, COUNT_IF(`qt_utilizacao_livre` = 0),
    'sg_unidade_medida_basica', 'string', COUNT_IF(`sg_unidade_medida_basica` IS NULL), COUNT_IF(`sg_unidade_medida_basica` IS NOT NULL AND lower(trim(`sg_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'vl_utilizacao_livre', 'double', COUNT_IF(`vl_utilizacao_livre` IS NULL), 0L, COUNT_IF(`vl_utilizacao_livre` = 0),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'qt_transito', 'decimal(13,3)', COUNT_IF(`qt_transito` IS NULL), 0L, COUNT_IF(`qt_transito` = 0),
    'vl_transito', 'double', COUNT_IF(`vl_transito` IS NULL), 0L, COUNT_IF(`vl_transito` = 0),
    'qt_controle_qualidade', 'decimal(13,3)', COUNT_IF(`qt_controle_qualidade` IS NULL), 0L, COUNT_IF(`qt_controle_qualidade` = 0),
    'vl_controle_qualidade', 'double', COUNT_IF(`vl_controle_qualidade` IS NULL), 0L, COUNT_IF(`vl_controle_qualidade` = 0),
    'qt_estoque_bloqueado', 'decimal(13,3)', COUNT_IF(`qt_estoque_bloqueado` IS NULL), 0L, COUNT_IF(`qt_estoque_bloqueado` = 0),
    'vl_estoque_bloqueado', 'double', COUNT_IF(`vl_estoque_bloqueado` IS NULL), 0L, COUNT_IF(`vl_estoque_bloqueado` = 0),
    'cod_estoque_especial', 'string', COUNT_IF(`cod_estoque_especial` IS NULL), COUNT_IF(`cod_estoque_especial` IS NOT NULL AND lower(trim(`cod_estoque_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_estoque_especial`) RLIKE '^0+([.,]0+)?$'),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L,
    'yearingest', 'string', COUNT_IF(`yearingest` IS NULL), COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`yearingest`) RLIKE '^0+([.,]0+)?$'),
    'monthingest', 'string', COUNT_IF(`monthingest` IS NULL), COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`monthingest`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                       THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                              AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [ ]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'),
card AS (
  SELECT stack(23,
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'tp_material', 'string', approx_count_distinct(`tp_material`),
    'tp_grupo_mercadorias', 'string', approx_count_distinct(`tp_grupo_mercadorias`),
    'nm_centro', 'string', approx_count_distinct(`nm_centro`),
    'ind_eliminacao_deposito', 'string', approx_count_distinct(`ind_eliminacao_deposito`),
    'tp_estoque_especial', 'string', approx_count_distinct(`tp_estoque_especial`),
    'qt_utilizacao_livre', 'decimal(13,3)', approx_count_distinct(`qt_utilizacao_livre`),
    'sg_unidade_medida_basica', 'string', approx_count_distinct(`sg_unidade_medida_basica`),
    'vl_utilizacao_livre', 'double', approx_count_distinct(`vl_utilizacao_livre`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'qt_transito', 'decimal(13,3)', approx_count_distinct(`qt_transito`),
    'vl_transito', 'double', approx_count_distinct(`vl_transito`),
    'qt_controle_qualidade', 'decimal(13,3)', approx_count_distinct(`qt_controle_qualidade`),
    'vl_controle_qualidade', 'double', approx_count_distinct(`vl_controle_qualidade`),
    'qt_estoque_bloqueado', 'decimal(13,3)', approx_count_distinct(`qt_estoque_bloqueado`),
    'vl_estoque_bloqueado', 'double', approx_count_distinct(`vl_estoque_bloqueado`),
    'cod_estoque_especial', 'string', approx_count_distinct(`cod_estoque_especial`),
    'dateingest', 'date', approx_count_distinct(`dateingest`),
    'yearingest', 'string', approx_count_distinct(`yearingest`),
    'monthingest', 'string', approx_count_distinct(`monthingest`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma, dentro do cenário.

In [ ]:
-- 8. DOMINIO DAS CATEGORICAS
(SELECT 'tp_material' AS coluna, CAST(`tp_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014' GROUP BY `tp_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_grupo_mercadorias' AS coluna, CAST(`tp_grupo_mercadorias` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014' GROUP BY `tp_grupo_mercadorias` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014' GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_estoque_especial' AS coluna, CAST(`tp_estoque_especial` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014' GROUP BY `tp_estoque_especial` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida_basica' AS coluna, CAST(`sg_unidade_medida_basica` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014' GROUP BY `sg_unidade_medida_basica` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014' GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_eliminacao_deposito' AS coluna, CAST(`ind_eliminacao_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014' GROUP BY `ind_eliminacao_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_estoque_especial' AS coluna, CAST(`cod_estoque_especial` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014' GROUP BY `cod_estoque_especial` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

Campos `double` exigem tolerância de 0,005 na comparação com o SAP.

In [ ]:
-- 9. PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(8,
    'qt_utilizacao_livre', 'decimal(13,3)', COUNT(`qt_utilizacao_livre`), CAST(MIN(`qt_utilizacao_livre`) AS DOUBLE), CAST(MAX(`qt_utilizacao_livre`) AS DOUBLE), CAST(AVG(`qt_utilizacao_livre`) AS DOUBLE), CAST(percentile_approx(`qt_utilizacao_livre`, 0.5) AS DOUBLE), COUNT_IF(`qt_utilizacao_livre` < 0), COUNT_IF(`qt_utilizacao_livre` = 0),
    'vl_utilizacao_livre', 'double', COUNT(`vl_utilizacao_livre`), CAST(MIN(`vl_utilizacao_livre`) AS DOUBLE), CAST(MAX(`vl_utilizacao_livre`) AS DOUBLE), CAST(AVG(`vl_utilizacao_livre`) AS DOUBLE), CAST(percentile_approx(`vl_utilizacao_livre`, 0.5) AS DOUBLE), COUNT_IF(`vl_utilizacao_livre` < 0), COUNT_IF(`vl_utilizacao_livre` = 0),
    'qt_transito', 'decimal(13,3)', COUNT(`qt_transito`), CAST(MIN(`qt_transito`) AS DOUBLE), CAST(MAX(`qt_transito`) AS DOUBLE), CAST(AVG(`qt_transito`) AS DOUBLE), CAST(percentile_approx(`qt_transito`, 0.5) AS DOUBLE), COUNT_IF(`qt_transito` < 0), COUNT_IF(`qt_transito` = 0),
    'vl_transito', 'double', COUNT(`vl_transito`), CAST(MIN(`vl_transito`) AS DOUBLE), CAST(MAX(`vl_transito`) AS DOUBLE), CAST(AVG(`vl_transito`) AS DOUBLE), CAST(percentile_approx(`vl_transito`, 0.5) AS DOUBLE), COUNT_IF(`vl_transito` < 0), COUNT_IF(`vl_transito` = 0),
    'qt_controle_qualidade', 'decimal(13,3)', COUNT(`qt_controle_qualidade`), CAST(MIN(`qt_controle_qualidade`) AS DOUBLE), CAST(MAX(`qt_controle_qualidade`) AS DOUBLE), CAST(AVG(`qt_controle_qualidade`) AS DOUBLE), CAST(percentile_approx(`qt_controle_qualidade`, 0.5) AS DOUBLE), COUNT_IF(`qt_controle_qualidade` < 0), COUNT_IF(`qt_controle_qualidade` = 0),
    'vl_controle_qualidade', 'double', COUNT(`vl_controle_qualidade`), CAST(MIN(`vl_controle_qualidade`) AS DOUBLE), CAST(MAX(`vl_controle_qualidade`) AS DOUBLE), CAST(AVG(`vl_controle_qualidade`) AS DOUBLE), CAST(percentile_approx(`vl_controle_qualidade`, 0.5) AS DOUBLE), COUNT_IF(`vl_controle_qualidade` < 0), COUNT_IF(`vl_controle_qualidade` = 0),
    'qt_estoque_bloqueado', 'decimal(13,3)', COUNT(`qt_estoque_bloqueado`), CAST(MIN(`qt_estoque_bloqueado`) AS DOUBLE), CAST(MAX(`qt_estoque_bloqueado`) AS DOUBLE), CAST(AVG(`qt_estoque_bloqueado`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_bloqueado`, 0.5) AS DOUBLE), COUNT_IF(`qt_estoque_bloqueado` < 0), COUNT_IF(`qt_estoque_bloqueado` = 0),
    'vl_estoque_bloqueado', 'double', COUNT(`vl_estoque_bloqueado`), CAST(MIN(`vl_estoque_bloqueado`) AS DOUBLE), CAST(MAX(`vl_estoque_bloqueado`) AS DOUBLE), CAST(AVG(`vl_estoque_bloqueado`) AS DOUBLE), CAST(percentile_approx(`vl_estoque_bloqueado`, 0.5) AS DOUBLE), COUNT_IF(`vl_estoque_bloqueado` < 0), COUNT_IF(`vl_estoque_bloqueado` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 10. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel extraído do SAP e compare linha a linha.
Divergência de total é o teste mais rápido para detectar registro faltando ou duplicado —
e cobre o ponto cego da contagem de linhas, que sozinha não prova nada.

In [ ]:
-- 10. TOTAIS PARA CONCILIACAO
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(8,
    'qt_utilizacao_livre', CAST(SUM(`qt_utilizacao_livre`) AS DOUBLE), CAST(ROUND(SUM(`qt_utilizacao_livre`), 2) AS STRING), COUNT(`qt_utilizacao_livre`),
    'vl_utilizacao_livre', CAST(SUM(`vl_utilizacao_livre`) AS DOUBLE), CAST(ROUND(SUM(`vl_utilizacao_livre`), 2) AS STRING), COUNT(`vl_utilizacao_livre`),
    'qt_transito', CAST(SUM(`qt_transito`) AS DOUBLE), CAST(ROUND(SUM(`qt_transito`), 2) AS STRING), COUNT(`qt_transito`),
    'vl_transito', CAST(SUM(`vl_transito`) AS DOUBLE), CAST(ROUND(SUM(`vl_transito`), 2) AS STRING), COUNT(`vl_transito`),
    'qt_controle_qualidade', CAST(SUM(`qt_controle_qualidade`) AS DOUBLE), CAST(ROUND(SUM(`qt_controle_qualidade`), 2) AS STRING), COUNT(`qt_controle_qualidade`),
    'vl_controle_qualidade', CAST(SUM(`vl_controle_qualidade`) AS DOUBLE), CAST(ROUND(SUM(`vl_controle_qualidade`), 2) AS STRING), COUNT(`vl_controle_qualidade`),
    'qt_estoque_bloqueado', CAST(SUM(`qt_estoque_bloqueado`) AS DOUBLE), CAST(ROUND(SUM(`qt_estoque_bloqueado`), 2) AS STRING), COUNT(`qt_estoque_bloqueado`),
    'vl_estoque_bloqueado', CAST(SUM(`vl_estoque_bloqueado`) AS DOUBLE), CAST(ROUND(SUM(`vl_estoque_bloqueado`), 2) AS STRING), COUNT(`vl_estoque_bloqueado`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 11.1 Datas em tipo nativo

In [ ]:
-- 11.1 DATAS NATIVAS
SELECT * FROM (
  SELECT stack(1,
    'dateingest', COUNT_IF(`dateingest` IS NULL), CAST(MIN(`dateingest`) AS STRING), CAST(MAX(`dateingest`) AS STRING), COUNT(DISTINCT `dateingest`), COUNT_IF(`dateingest` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 12. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [ ]:
-- 12. CODIGOS
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(3,
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_deposito', 'string', COUNT_IF(CAST(`cod_deposito` AS STRING) IS NULL OR trim(CAST(`cod_deposito` AS STRING)) = ''), MIN(length(trim(CAST(`cod_deposito` AS STRING)))), MAX(length(trim(CAST(`cod_deposito` AS STRING)))), COUNT_IF(trim(CAST(`cod_deposito` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_deposito` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_deposito` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 13. Chaves normalizadas para join com o SAP

Lista das chaves já **sem zeros à esquerda**, prontas para colar no Excel
e cruzar com o extrato do SAP via PROCV/ÍNDICE.

Baixe como CSV e use para identificar registros presentes de um lado e ausentes do outro.

In [ ]:
-- 13. CHAVES NORMALIZADAS (para cruzar com o SAP)
SELECT DISTINCT
       regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS `cod_material_norm`,
       regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') AS `cod_centro_norm`
FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
ORDER BY 1, 2;

## 14. Checksum de linha

Gera uma impressão digital de cada linha. Dois usos:

- **Contar linhas realmente distintas** — se `linhas` for maior que `linhas_unicas`,
  existem registros 100% idênticos (duplicata real)
- **Comparação rápida** — aplicando a mesma concatenação no SAP, dá para achar
  divergências sem comparar campo a campo

In [ ]:
-- 14. CHECKSUM DE LINHA
SELECT COUNT(*)                                                 AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`ind_eliminacao_deposito` AS STRING), ''), COALESCE(CAST(`tp_estoque_especial` AS STRING), ''), COALESCE(CAST(`qt_utilizacao_livre` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`vl_utilizacao_livre` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`qt_transito` AS STRING), ''), COALESCE(CAST(`vl_transito` AS STRING), ''), COALESCE(CAST(`qt_controle_qualidade` AS STRING), ''), COALESCE(CAST(`vl_controle_qualidade` AS STRING), ''), COALESCE(CAST(`qt_estoque_bloqueado` AS STRING), ''), COALESCE(CAST(`vl_estoque_bloqueado` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))             AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`ind_eliminacao_deposito` AS STRING), ''), COALESCE(CAST(`tp_estoque_especial` AS STRING), ''), COALESCE(CAST(`qt_utilizacao_livre` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`vl_utilizacao_livre` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`qt_transito` AS STRING), ''), COALESCE(CAST(`vl_transito` AS STRING), ''), COALESCE(CAST(`qt_controle_qualidade` AS STRING), ''), COALESCE(CAST(`vl_controle_qualidade` AS STRING), ''), COALESCE(CAST(`qt_estoque_bloqueado` AS STRING), ''), COALESCE(CAST(`vl_estoque_bloqueado` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))  AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`ind_eliminacao_deposito` AS STRING), ''), COALESCE(CAST(`tp_estoque_especial` AS STRING), ''), COALESCE(CAST(`qt_utilizacao_livre` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`vl_utilizacao_livre` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`qt_transito` AS STRING), ''), COALESCE(CAST(`vl_transito` AS STRING), ''), COALESCE(CAST(`qt_controle_qualidade` AS STRING), ''), COALESCE(CAST(`vl_controle_qualidade` AS STRING), ''), COALESCE(CAST(`qt_estoque_bloqueado` AS STRING), ''), COALESCE(CAST(`vl_estoque_bloqueado` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014';

## 15. Amostra de linhas completas

In [ ]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
ORDER BY `cod_material`, `cod_deposito`
LIMIT 20;

## 16. Distribuição interna do centro 4014

Como o volume se reparte dentro do cenário. Útil para conferir se o extrato do SAP
tem a mesma composição.

In [ ]:
-- 16. DISTRIBUICAO POR cod_deposito
SELECT COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)') AS `cod_deposito`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR tp_material
SELECT COALESCE(NULLIF(trim(CAST(`tp_material` AS STRING)), ''), '(vazio)') AS `tp_material`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_material` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR tp_grupo_mercadorias
SELECT COALESCE(NULLIF(trim(CAST(`tp_grupo_mercadorias` AS STRING)), ''), '(vazio)') AS `tp_grupo_mercadorias`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_grupo_mercadorias` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [ ]:
-- 17. FRESHNESS
SELECT `dateingest` AS data_carga, COUNT(*) AS linhas
FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
GROUP BY `dateingest`
ORDER BY data_carga DESC
LIMIT 30;

## 18. Análises específicas — MB52

### 18.1 Colapso de depósito no centro 4014

No SAP a granularidade é material + centro + depósito + tipo de estoque especial.

In [ ]:
-- 18.1 DEPOSITOS POR MATERIAL
SELECT qt_depositos, COUNT(*) AS materiais, SUM(linhas - 1) AS linhas_excedentes
FROM (
  SELECT cod_material, COUNT(DISTINCT cod_deposito) AS qt_depositos, COUNT(*) AS linhas
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
  GROUP BY cod_material
)
GROUP BY qt_depositos
ORDER BY qt_depositos;

In [ ]:
-- 18.1b MATERIAIS COM MAIS DEPOSITOS
SELECT cod_material,
       COUNT(DISTINCT cod_deposito) AS qt_depositos,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(cod_deposito))) AS depositos
FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
GROUP BY cod_material
HAVING COUNT(DISTINCT cod_deposito) > 1
ORDER BY qt_depositos DESC
LIMIT 25;

### 18.2 Coerência quantidade × valor

In [ ]:
-- 18.2 QUANTIDADE x VALOR
SELECT 'qt_utilizacao_livre / vl_utilizacao_livre' AS par,
       COUNT_IF(`qt_utilizacao_livre` <> 0 OR `vl_utilizacao_livre` <> 0) AS linhas_com_estoque,
       COUNT_IF(`qt_utilizacao_livre` = 0 AND `vl_utilizacao_livre` <> 0) AS qtd_zero_valor_nao,
       COUNT_IF(`qt_utilizacao_livre` <> 0 AND `vl_utilizacao_livre` = 0) AS qtd_nao_valor_zero,
       COUNT_IF(`qt_utilizacao_livre` < 0) AS qtd_negativa
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
UNION ALL
SELECT 'qt_transito / vl_transito' AS par,
       COUNT_IF(`qt_transito` <> 0 OR `vl_transito` <> 0) AS linhas_com_estoque,
       COUNT_IF(`qt_transito` = 0 AND `vl_transito` <> 0) AS qtd_zero_valor_nao,
       COUNT_IF(`qt_transito` <> 0 AND `vl_transito` = 0) AS qtd_nao_valor_zero,
       COUNT_IF(`qt_transito` < 0) AS qtd_negativa
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
UNION ALL
SELECT 'qt_controle_qualidade / vl_controle_qualidade' AS par,
       COUNT_IF(`qt_controle_qualidade` <> 0 OR `vl_controle_qualidade` <> 0) AS linhas_com_estoque,
       COUNT_IF(`qt_controle_qualidade` = 0 AND `vl_controle_qualidade` <> 0) AS qtd_zero_valor_nao,
       COUNT_IF(`qt_controle_qualidade` <> 0 AND `vl_controle_qualidade` = 0) AS qtd_nao_valor_zero,
       COUNT_IF(`qt_controle_qualidade` < 0) AS qtd_negativa
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'
UNION ALL
SELECT 'qt_estoque_bloqueado / vl_estoque_bloqueado' AS par,
       COUNT_IF(`qt_estoque_bloqueado` <> 0 OR `vl_estoque_bloqueado` <> 0) AS linhas_com_estoque,
       COUNT_IF(`qt_estoque_bloqueado` = 0 AND `vl_estoque_bloqueado` <> 0) AS qtd_zero_valor_nao,
       COUNT_IF(`qt_estoque_bloqueado` <> 0 AND `vl_estoque_bloqueado` = 0) AS qtd_nao_valor_zero,
       COUNT_IF(`qt_estoque_bloqueado` < 0) AS qtd_negativa
  FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014';

## 19. EXTRAÇÃO COMPLETA — centro 4014

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for muito grande, use a célula 19.1.

In [ ]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.tbl_ds_log_mb52
WHERE `cod_centro` = '4014'
ORDER BY `cod_material`, `cod_deposito`;

### 19.1 Alternativa para volume grande _(opcional)_

Descomente para gravar o resultado numa tabela própria e exportar de lá sem limite de tela.

In [ ]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_mb52_4014 AS
-- SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_mb52_4014;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

Bloco final. **Copie esta saída** e envie ao agente junto com o notebook.

In [ ]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito` FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito + tp_estoque_especial' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial` FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito + tp_estoque_especial + dateingest' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial`, `dateingest` FROM dev_procurement.corp_curated.tbl_ds_log_mb52 WHERE `cod_centro` = '4014')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'MB52' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'dev_procurement.corp_curated.tbl_ds_log_mb52'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4014'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '23'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4014 no Datalake.
2. Extrair a mesma transação no SAP com o filtro `centro = 4014`, **todas as abas**.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Zeros à esquerda normalizados nos dois lados (seção 12)
- [ ] Formato de data normalizado (seção 11)
- [ ] Totais numéricos conferidos (seção 10)
- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP (seção 6)
